<img src="../../docs/site/_static/shot-logo.png" alt="SHOT" width="240">

# Lazy nonlinear constraints with SHOTpy callbacks

This example solves a convex MINLP where some nonlinear constraints are **not** given to SHOT. They are known only
to a black box, a Python function that stands in for external code such as a simulator. Given a point, the black box
says which of its constraints are violated, by how much, and returns a linear cut for each that removes the point
but no feasible solution. The constraints are enforced lazily with two callbacks:

1. `PrimalCandidateCheck` rejects every primal candidate that the black box reports as infeasible, so SHOT never
   accepts a solution that is infeasible for the full problem.
2. `HyperplaneSelection` asks the black box about the solution points of the dual (MIP) problems and adds its cuts,
   so the dual problems converge to the full problem and the dual bound stays valid.

The callbacks know nothing about how the black box evaluates its constraints or generates its cuts. This is useful
when a constraint is expensive to evaluate, comes from a simulator or another program, or is only needed near the
optimum.

**The problem** is [`ex1223b` from MINLPLib](https://www.minlplib.org/ex1223b.html): three continuous and four binary
variables, a convex nonlinear objective, five linear constraints and four convex quadratic constraints. Its optimal
objective value is 4.5795824.

$$
\begin{aligned}
\min\;& (b_4-1)^2 + (b_5-2)^2 + (b_6-1)^2 - \log(1+b_7) + (x_1-1)^2 + (x_2-2)^2 + (x_3-3)^2 \\
\text{s.t.}\;& x_1 + x_2 + x_3 + b_4 + b_5 + b_6 \le 5 && (e_1)\\
& b_6^2 + x_1^2 + x_2^2 + x_3^2 \le 5.5 && (e_2)\\
& x_1 + b_4 \le 1.2,\quad x_2 + b_5 \le 1.8,\quad x_3 + b_6 \le 2.5,\quad x_1 + b_7 \le 1.2 && (e_3\text{–}e_6)\\
& b_5^2 + x_2^2 \le 1.64 && (e_7)\\
& b_6^2 + x_3^2 \le 4.25 && (e_8)\\
& b_5^2 + x_3^2 \le 4.64 && (e_9)\\
& x_1, x_2, x_3 \in [0, 10],\quad b_4, b_5, b_6, b_7 \in \{0, 1\}
\end{aligned}
$$

The constraints $e_7$, $e_8$ and $e_9$ are left out of the model given to SHOT and are known only to the black box.

**Rejecting candidates and adding cuts do different things.** Rejection alone keeps infeasible solutions out of the
primal bound, but the dual problems still lack the omitted constraints: their bound is the bound of the reduced
problem, and the gap never closes. Cuts alone make the dual problems converge, but SHOT's primal heuristics, which only
know the reduced problem, would still accept points that violate the omitted constraints. The example uses both.

The notebook makes three independent runs with the same settings: the full problem, the reduced problem without
callbacks, and the reduced problem with the callbacks.

## Settings and dependencies

The example needs SHOTpy built with HiGHS, and has been tested with HiGHS 1.15 and Ipopt as the NLP solver for the
fixed-integer problems. Cbc gives the same results. With HiGHS and Cbc, SHOT always uses its multi-tree strategy,
which is what the example is written for. With Gurobi or CPLEX, SHOT may instead pass the quadratic reduced problem
directly to the MIP solver, and the single-tree strategy and those solvers are not covered here. Matplotlib is
optional and is only used for the final figure.

The settings are SHOT's defaults, except:

* `Dual.TreeStrategy = 0` selects the multi-tree strategy explicitly.
* `Dual.MIP.NumberOfThreads = 1` makes the runs reproducible.
* `Output.Console.LogLevel = 6` turns off SHOT's console output, since the callbacks log what is of interest here.

One default matters for omitted constraints, and is kept on purpose: `Dual.HyperplaneCuts.UseIntegerCuts` is off.
An integer cut would exclude an integer assignment for good, but that a point of the reduced problem violates an
omitted constraint does not mean that its integer assignment is infeasible for the full problem.

SHOT's fixed-integer NLP problems are built from the problem SHOT is given, so they contain neither the omitted
constraints nor the cuts from the callbacks. With the integer variables fixed, the NLP problem is the same every time
and has the same solution, which can violate the omitted constraints. How often SHOT solves it, e.g., with
`Primal.FixedInteger.OnlyUniqueIntegerCombinations` turned off, does not change that. Run 4 shows how a black-box
heuristic can supply primal solutions instead.

**Tolerances.** A candidate is rejected when the black box reports a violation larger than SHOT's own tolerance for
accepting nonlinear constraints, `Primal.Tolerance.NonlinearConstraint` (1e-5). A cut is added whenever the black box
reports a violation larger than `1e-9`, a much smaller tolerance. Every rejected candidate is thus also given a cut.
There is no range of violations where candidates are rejected but no cut can be generated.

In [ ]:
import math
import random

import SHOTpy

# The MIP solver the example has been tested with. With HiGHS or Cbc, SHOT uses the multi-tree strategy
MIP_SOLVER = SHOTpy.MIPSolver.Highs

# The names of the MIP solvers as they are written
SOLVER_NAMES = {SHOTpy.MIPSolver.Highs: "HiGHS", SHOTpy.MIPSolver.Cbc: "Cbc", SHOTpy.MIPSolver.Gurobi: "Gurobi",
                SHOTpy.MIPSolver.Cplex: "CPLEX"}

if MIP_SOLVER not in SHOTpy.getSupportedMIPSolvers():
    raise RuntimeError(f"This SHOTpy build does not include {SOLVER_NAMES[MIP_SOLVER]}, which the example has been "
                       "tested with")

SETTINGS = {
    "Output.Console.LogLevel": 6,  # Off
    "Dual.MIP.Solver": int(MIP_SOLVER),
    "Dual.TreeStrategy": 0,  # Multi-tree
    "Dual.MIP.NumberOfThreads": 1,
}

# A candidate is rejected when an omitted constraint is violated by more than SHOT's own tolerance for nonlinear
# constraints, and a cut is added for any violation larger than round-off
FEASIBILITY_TOLERANCE = SHOTpy.Solver().getDoubleSetting("Primal.Tolerance.NonlinearConstraint")
SEPARATION_TOLERANCE = 1e-9
assert SEPARATION_TOLERANCE < FEASIBILITY_TOLERANCE

# SHOT's tolerances for integrality and for terminating on the objective gap, used when the results are checked
INTEGER_TOLERANCE = SHOTpy.Solver().getDoubleSetting("Primal.Tolerance.Integer")
GAP_TOLERANCE = SHOTpy.Solver().getDoubleSetting("Termination.ObjectiveGap.Absolute")

# The optimal objective value of ex1223b given by MINLPLib
KNOWN_OPTIMUM = 4.5795824

print(f"MIP solver: {SOLVER_NAMES[MIP_SOLVER]}, feasibility tolerance: {FEASIBILITY_TOLERANCE:g}, "
      f"separation tolerance: {SEPARATION_TOLERANCE:g}")

## The model

The constraints are given as data, once, and are used both to build the SHOTpy model and inside the black box below,
so the two cannot drift apart. Every constraint of `ex1223b` has the form

$$
\sum_i a_i x_i + \sum_{j \in S} x_j^2 \le u,
$$

so a constraint is given by its linear coefficients $a$, its squared variables $S$ and its upper bound $u$.

`build_solver(omitted)` builds a new solver and model each time, and leaves the omitted constraints out before the
model is finalized and given to SHOT.

In [ ]:
# The variables in the order of their indexes in SHOT, with their bounds
VARIABLES = {
    "x1": (SHOTpy.VariableType.Real, 0.0, 10.0),
    "x2": (SHOTpy.VariableType.Real, 0.0, 10.0),
    "x3": (SHOTpy.VariableType.Real, 0.0, 10.0),
    "b4": (SHOTpy.VariableType.Binary, 0.0, 1.0),
    "b5": (SHOTpy.VariableType.Binary, 0.0, 1.0),
    "b6": (SHOTpy.VariableType.Binary, 0.0, 1.0),
    "b7": (SHOTpy.VariableType.Binary, 0.0, 1.0),
}
INDEX = {name: index for index, name in enumerate(VARIABLES)}

# Each constraint is sum(a_i x_i) + sum(x_j^2 for j in squared) <= upper: (linear coefficients a, squared, upper)
CONSTRAINTS = {
    "e1": ({"x1": 1, "x2": 1, "x3": 1, "b4": 1, "b5": 1, "b6": 1}, [], 5.0),
    "e2": ({}, ["b6", "x1", "x2", "x3"], 5.5),
    "e3": ({"x1": 1, "b4": 1}, [], 1.2),
    "e4": ({"x2": 1, "b5": 1}, [], 1.8),
    "e5": ({"x3": 1, "b6": 1}, [], 2.5),
    "e6": ({"x1": 1, "b7": 1}, [], 1.2),
    "e7": ({}, ["b5", "x2"], 1.64),
    "e8": ({}, ["b6", "x3"], 4.25),
    "e9": ({}, ["b5", "x3"], 4.64),
}

# The constraints left out of the model and known only to the black box. Any subset works, since they are all convex
OMITTED = ("e7", "e8", "e9")


def build_solver(omitted=()):
    unknown = set(omitted) - set(CONSTRAINTS)

    if unknown:
        raise ValueError(f"Unknown constraints {sorted(unknown)}")

    solver = SHOTpy.Solver()

    for name, value in SETTINGS.items():
        solver.updateSetting(name, value)

    problem = SHOTpy.Problem(solver)
    problem.name = "ex1223b"

    x = {name: problem.addVariable(name, vtype, lower, upper) for name, (vtype, lower, upper) in VARIABLES.items()}

    problem.setObjective((x["b4"] - 1.0)**2 + (x["b5"] - 2.0)**2 + (x["b6"] - 1.0)**2 - SHOTpy.log(1.0 + x["b7"])
                         + (x["x1"] - 1.0)**2 + (x["x2"] - 2.0)**2 + (x["x3"] - 3.0)**2,
                         SHOTpy.ObjectiveDirection.Minimize)

    for name, (linear, squared, upper) in CONSTRAINTS.items():
        if name not in omitted:
            body = sum(a * x[v] for v, a in linear.items()) + sum(x[v]**2 for v in squared)
            problem.addConstraint(body <= upper, name)

    problem.finalize()
    assert solver.setProblem(problem)

    return solver, problem

## The black box

`black_box(point)` is everything the rest of the notebook knows about the omitted constraints. Given a point in the
original variables `x1, x2, x3, b4, b5, b6, b7`, it returns one entry for each omitted constraint that is violated at
the point, and nothing when the point is feasible. An entry has

* `constraint`: the name of the constraint,
* `violation`: how much the constraint is violated,
* `coefficients` and `rhs`: a cut $\sum_i \texttt{coefficients}[i]\, x_i \le \texttt{rhs}$ that the point violates,
  but that every point fulfilling the constraint satisfies.

The callbacks rely only on this contract. Replace the function with any other code that keeps it, e.g., a call to a
simulator, and the callbacks work unchanged.

**Inside the black box** are extended cutting plane (ECP) cuts. Each constraint is written as $g(x) \le 0$ with
$g(x) = \sum_i a_i x_i + \sum_{j \in S} x_j^2 - u$. Since $g$ is convex, its linearization at any point $\bar x$
underestimates it:

$$
g(\bar x) + \nabla g(\bar x)^\top (x - \bar x) \le g(x).
$$

Every $x$ with $g(x) \le 0$ therefore fulfils the cut

$$
\nabla g(\bar x)^\top x \le \nabla g(\bar x)^\top \bar x - g(\bar x),
$$

and at $\bar x$ itself the cut is violated by exactly $g(\bar x) > 0$.

In [ ]:
def _residual(name, point):
    # g(x) = f(x) - upper, which is positive when the constraint is violated
    linear, squared, upper = CONSTRAINTS[name]
    return (sum(a * point[INDEX[v]] for v, a in linear.items()) + sum(point[INDEX[v]] ** 2 for v in squared) - upper)


def _gradient(name, point):
    # The gradient of g as {variable index: value}
    linear, squared, _ = CONSTRAINTS[name]
    result = {}

    for v, a in linear.items():
        result[INDEX[v]] = result.get(INDEX[v], 0.0) + a

    for v in squared:
        result[INDEX[v]] = result.get(INDEX[v], 0.0) + 2.0 * point[INDEX[v]]

    return result


def black_box(point):
    # For each omitted constraint violated at the point: its violation and the ECP cut that removes the point
    result = []

    for name in OMITTED:
        violation = _residual(name, point)

        if violation > 0.0:
            coefficients = _gradient(name, point)
            rhs = sum(a * point[i] for i, a in coefficients.items()) - violation
            result.append({"constraint": name, "violation": violation, "coefficients": coefficients, "rhs": rhs})

    return result

### Testing the black box

The test treats the black box as a black box too. It compares what it reports with SHOT's own evaluation of the full
model, at random points within the variable bounds. It checks that the black box reports exactly the violated
constraints with the right violations, that each cut removes the point it was generated in, and that no cut removes
any of the sampled points that fulfil its constraint. Since the callbacks rely on these properties, this is the
place to catch a wrong index or coefficient.

In [ ]:
def cut_violation(cut, point):
    # How much the point violates the cut sum(coefficients[i] * x[i]) <= rhs
    return sum(a * point[i] for i, a in cut["coefficients"].items()) - cut["rhs"]


_, full_problem = build_solver()
assert [v.name for v in full_problem.allVariables] == list(VARIABLES)
shot_constraints = {c.name: c for c in full_problem.numericConstraints}


def shot_residual(name, point):
    constraint = shot_constraints[name]
    return constraint.calculateFunctionValue(point) - constraint.valueRHS


rng = random.Random(1223)
points = [[rng.uniform(lower, upper) for _, lower, upper in VARIABLES.values()] for _ in range(500)]
feasible = {name: [p for p in points if shot_residual(name, p) <= 0.0] for name in OMITTED}
number_of_cuts = 0

for point in points:
    reported = {cut["constraint"]: cut for cut in black_box(point)}

    for name in OMITTED:
        value = shot_residual(name, point)

        # Exactly the violated constraints are reported, with the violation SHOT calculates
        assert (name in reported) == (value > 0.0), name

        if name in reported:
            cut = reported[name]
            assert math.isclose(cut["violation"], value, rel_tol=1e-12, abs_tol=1e-12)

            # The cut removes the point, and none of the sampled points that fulfil the constraint
            assert cut_violation(cut, point) > 0.0
            assert all(cut_violation(cut, p) <= 1e-9 for p in feasible[name])
            number_of_cuts += 1

print(f"Checked {number_of_cuts} cuts at {len(points)} points against "
      + ", ".join(f"{len(feasible[name])} points fulfilling {name}" for name in OMITTED))

## Run 1: the full problem

The reference: SHOT solves the problem with all nine constraints. The black box is only asked whether the solution
fulfils the omitted constraints.

In [ ]:
def omitted_violation(point):
    # The largest violation of the omitted constraints reported by the black box, zero when there is none
    return max((cut["violation"] for cut in black_box(point)), default=0.0)


def run_summary(label, solver, handler=None):
    solution = solver.getPrimalSolution() if solver.hasPrimalSolution() else None

    return {
        "run": label,
        "primal": solver.getPrimalBound(),
        "dual": solver.getGlobalDualBound(),
        "gap": solver.getAbsoluteObjectiveGap(),
        "status": solver.getModelReturnStatus().name,
        "termination": solver.getTerminationReason().name,
        "point": list(solution.point) if solution else None,
        "omitted violation": omitted_violation(solution.point) if solution else math.nan,
        "rejected": len(handler.rejected) if handler else 0,
        "cuts": len(handler.cuts) if handler else 0,
    }


def print_point(summary):
    print(f"{summary['run']}: objective {summary['primal']:.7f}, dual bound {summary['dual']:.7f}, "
          f"{summary['status']}, terminated on {summary['termination']}")
    print("  point:     " + ", ".join(f"{name} = {value:.4f}" for name, value in zip(VARIABLES, summary["point"])))
    # Constraints that are active at the point can be reported as violated by round-off
    violated = [cut for cut in black_box(summary["point"]) if cut["violation"] > FEASIBILITY_TOLERANCE]
    print("  black box: " + (", ".join(f"{cut['constraint']} violated by {cut['violation']:.2e}" for cut in violated)
                             or "no violation above the feasibility tolerance"))


reference_solver, _ = build_solver()
assert reference_solver.solveProblem()
reference = run_summary("full problem", reference_solver)
print_point(reference)

assert reference["status"] == "OptimalGlobal"
assert abs(reference["primal"] - KNOWN_OPTIMUM) <= GAP_TOLERANCE
assert reference["omitted violation"] <= FEASIBILITY_TOLERANCE

## Run 2: the reduced problem without callbacks

The omitted constraints are not in the problem SHOT gets. SHOT solves the reduced problem to optimality, which gives
a lower objective value than the full problem, at a point the black box reports as infeasible.

In [ ]:
reduced_solver, reduced_problem = build_solver(OMITTED)

# The omitted constraints are not in the problem, and everything else is
names = [c.name for c in reduced_problem.numericConstraints]
assert names == [name for name in CONSTRAINTS if name not in OMITTED], names
assert [v.name for v in reduced_problem.allVariables] == list(VARIABLES)
assert all(math.isclose(reduced_problem.objectiveFunction.calculateValue(p),
                        full_problem.objectiveFunction.calculateValue(p)) for p in points)
print(f"Constraints given to SHOT: {', '.join(names)}")

assert reduced_solver.solveProblem()
reduced = run_summary("reduced problem", reduced_solver)
print_point(reduced)

# The reduced problem is solved, but its solution is infeasible for the full problem
assert reduced["status"] == "OptimalGlobal"
assert reduced["omitted violation"] > FEASIBILITY_TOLERANCE
assert reduced["primal"] < reference["primal"] - GAP_TOLERANCE

## Run 3: the reduced problem with lazy constraints

`LazyConstraints` holds the callbacks and what they log. It is given the black box and uses nothing else about the
omitted constraints. Callback contexts can only be used while the callback runs, so the callbacks copy the values
they need into ordinary Python lists:

* `check_candidate` (`PrimalCandidateCheck`) asks the black box about each primal candidate and calls
  `ctx.rejectCandidate()` if a constraint is violated by more than the feasibility tolerance. The rejected point is
  also queued for the cut generation, so points from SHOT's primal heuristics, not only the solution points of the
  dual problems, are cut off.
* `add_cuts` (`HyperplaneSelection`) asks the black box about the solution points of the last dual problem and the
  queued points, and adds each cut it returns as a global `ExternalHyperplane`. Since the cut generation is unknown,
  only the cut itself is checked: its coefficients must be finite, and it must remove the point it was generated in.
* `record_solution` (`NewPrimalSolution`) logs each solution SHOT accepts. It runs after a solution has been accepted,
  so it cannot reject one; it is used to check that every accepted solution satisfies the omitted constraints.
* `record_bounds` (`TerminationCheck`) logs the bounds in each iteration for the figure.

The solution points of the dual problems are in the variables of the problem SHOT has reformulated, which adds
auxiliary variables after the original ones. The black box is given the first seven values, after checking that they
are the original variables. The indexes of the cuts are then the indexes of the original variables. The callbacks do
not call any methods of the solver.

In [ ]:
def to_hyperplane(cut, point):
    # How the cut was generated is unknown, so only the cut itself can be checked
    if not all(math.isfinite(a) for a in cut["coefficients"].values()) or not math.isfinite(cut["rhs"]):
        raise ValueError(f"The cut for {cut['constraint']} at {point} is not finite")

    if cut_violation(cut, point) <= 0.0:
        raise ValueError(f"The cut for {cut['constraint']} does not remove the point {point} it was generated in")

    hyperplane = SHOTpy.ExternalHyperplane()
    hyperplane.variableIndexes = list(cut["coefficients"])
    hyperplane.variableCoefficients = list(cut["coefficients"].values())
    hyperplane.rhsValue = cut["rhs"]
    hyperplane.isGlobal = True  # The black box promises that the cut removes no feasible solution
    hyperplane.source = SHOTpy.HyperplaneSource.External
    hyperplane.description = f"Black-box cut for {cut['constraint']}"

    return hyperplane


class LazyConstraints:
    def __init__(self, oracle):
        self.oracle = oracle  # The black box
        self.rejected = []    # Rejected primal candidates
        self.accepted = []    # Primal solutions accepted by SHOT
        self.cuts = []        # Cuts given to SHOT
        self.dual_points = [] # (iteration, the largest violation at the solution points of its dual problem)
        self.bounds = []      # (iteration, primal bound, dual bound)
        self.queued = []      # Rejected points that have not been cut off yet

    def register(self, solver):
        solver.registerCallback(SHOTpy.CallbackLocation.PrimalCandidateCheck, self.check_candidate)
        solver.registerCallback(SHOTpy.CallbackLocation.HyperplaneSelection, self.add_cuts)
        solver.registerCallback(SHOTpy.CallbackLocation.NewPrimalSolution, self.record_solution)
        solver.registerCallback(SHOTpy.CallbackLocation.TerminationCheck, self.record_bounds)

    def largest_violation(self, point):
        return max((cut["violation"] for cut in self.oracle(point)), default=0.0)

    def check_candidate(self, ctx):
        point = list(ctx.point)
        violated = {cut["constraint"]: cut["violation"] for cut in self.oracle(point)
                    if cut["violation"] > FEASIBILITY_TOLERANCE}

        if violated:
            ctx.rejectCandidate()
            self.queued.append(point)
            self.rejected.append({"iteration": ctx.iterationNumber, "objective": ctx.objectiveValue,
                                  "source": ctx.source.name, "violated": violated,
                                  "finalizing": ctx.isFinalizing})

    def record_solution(self, ctx):
        self.accepted.append({"iteration": ctx.iterationNumber, "objective": ctx.objectiveValue,
                              "source": ctx.source.name, "violation": self.largest_violation(list(ctx.point))})

    def record_bounds(self, ctx):
        self.bounds.append((ctx.iterationNumber, ctx.primalBound, ctx.dualBound))

    def add_cuts(self, ctx):
        reformulated = [v.name for v in ctx.reformulatedProblem.allVariables]

        if reformulated[:len(VARIABLES)] != list(VARIABLES):
            raise RuntimeError(f"The original variables are not first in the reformulated problem: {reformulated}")

        dual_points = [list(p.point)[:len(VARIABLES)] for p in ctx.solutionPoints]

        # The solution points are those of the dual problem of the previous iteration
        if dual_points:
            self.dual_points.append((ctx.solutionPoints[0].iterFound,
                                     max(self.largest_violation(p) for p in dual_points)))

        points, self.queued = dual_points + self.queued, []
        seen = set()

        for point in points:
            # Only exact repetitions are skipped, since nearby points can need different cuts
            if tuple(point) in seen:
                continue

            seen.add(tuple(point))

            for cut in self.oracle(point):
                if cut["violation"] > SEPARATION_TOLERANCE:
                    ctx.addHyperplane(to_hyperplane(cut, point))
                    self.cuts.append({"iteration": ctx.iterationNumber, "cut": cut, "point": point})


lazy_solver, _ = build_solver(OMITTED)
handler = LazyConstraints(black_box)
handler.register(lazy_solver)

assert lazy_solver.solveProblem()
lazy = run_summary("reduced problem with callbacks", lazy_solver, handler)
print_point(lazy)
print(f"  {len(handler.rejected)} candidates rejected, {len(handler.cuts)} cuts submitted")

## Results

The table compares the three runs. The reduced problem without callbacks gives a lower, infeasible objective value,
while the run with callbacks finds the optimum of the full problem and proves it with a valid dual bound, without
ever having the omitted constraints in the model.

In [ ]:
runs = [reference, reduced, lazy]

print(f"{'run':<32}{'primal':>11}{'dual':>11}{'gap':>10}  {'status':<15}{'termination':<13}"
      f"{'max violation':>14}{'rejected':>10}{'cuts':>6}")

for r in runs:
    print(f"{r['run']:<32}{r['primal']:>11.6f}{r['dual']:>11.6f}{r['gap']:>10.2e}  {r['status']:<15}"
          f"{r['termination']:<13}{r['omitted violation']:>14.2e}{r['rejected']:>10}{r['cuts']:>6}")

print("\nRejected candidates:")
for r in handler.rejected:
    violated = ", ".join(f"{name} by {value:.2e}" for name, value in r["violated"].items())
    print(f"  iteration {r['iteration']:>2}: objective {r['objective']:.6f} from {r['source']:<22} violates {violated}")

print("\nAccepted primal solutions:")
for a in handler.accepted:
    print(f"  iteration {a['iteration']:>2}: objective {a['objective']:.6f} from {a['source']:<22} "
          f"largest violation {a['violation']:.2e}")

print("\nCuts added to the dual problem of each iteration:")
per_iteration = {}
for c in handler.cuts:
    per_iteration.setdefault(c["iteration"], []).append(c["cut"]["constraint"])
for iteration, names in per_iteration.items():
    print(f"  iteration {iteration:>2}: {', '.join(names)}")

The rejected candidates show where SHOT's primal heuristics would otherwise have accepted infeasible solutions: points
of the dual problems from the MIP solution pool, points from the rootsearch, and the fixed-integer NLP solution of the
optimal integer assignment, which is the optimum of the reduced problem. The optimum of the full problem is accepted
once the cuts make a solution of a dual problem feasible.

### Validation

The final solution is checked independently of the black box and the callback log, against SHOT's evaluation of a
newly built full problem: variable bounds, integrality and all nine constraints. The other checks cover what the
callbacks did.

In [ ]:
point = lazy["point"]
_, check_problem = build_solver()

# Bounds and integrality
for variable, value in zip(check_problem.allVariables, point):
    assert variable.lowerBound - INTEGER_TOLERANCE <= value <= variable.upperBound + INTEGER_TOLERANCE, variable.name
    if VARIABLES[variable.name][0] == SHOTpy.VariableType.Binary:
        assert abs(value - round(value)) <= INTEGER_TOLERANCE, variable.name

# All constraints of the full problem, as evaluated by SHOT
for constraint in check_problem.numericConstraints:
    assert constraint.calculateFunctionValue(point) - constraint.valueRHS <= FEASIBILITY_TOLERANCE, constraint.name

# The callbacks rejected candidates and gave SHOT cuts, and every cut removed the point it was generated in
assert handler.rejected and handler.cuts
assert all(cut_violation(c["cut"], c["point"]) > SEPARATION_TOLERANCE for c in handler.cuts)

# Every solution SHOT accepted satisfies the omitted constraints
assert all(a["violation"] <= FEASIBILITY_TOLERANCE for a in handler.accepted)

# The optimum is proven: the gap is closed, and the dual bound is valid for the full problem
assert lazy["status"] == "OptimalGlobal"
assert lazy["termination"] in ("AbsoluteGap", "RelativeGap")
assert lazy["gap"] <= GAP_TOLERANCE
assert lazy["dual"] <= KNOWN_OPTIMUM + 1e-6

# The objective value agrees with the full problem and MINLPLib within the termination tolerance on the gap
assert abs(lazy["primal"] - reference["primal"]) <= GAP_TOLERANCE
assert abs(lazy["primal"] - KNOWN_OPTIMUM) <= GAP_TOLERANCE

print("All checks passed")

### Convergence

The figure shows the bounds of the run with callbacks, and the largest violation the black box reports at the
solution points of the dual problem of each iteration. The cuts for the points of one iteration are added to the dual
problem of the next. The violation falls as the cuts accumulate, and the dual bound rises to the optimum of the full
problem rather than stopping at the optimum of the reduced problem.

The figure has two panels over the iterations. The upper panel shows the primal bound falling and the dual bound
rising until they meet at the optimum of the full problem, above the optimum of the reduced problem. The lower panel
shows the largest violation at the dual solution points on a log scale, with open dots for iterations where the
omitted constraints are fulfilled.

In [ ]:
try:
    import matplotlib.pyplot as plt
    from matplotlib import font_manager
except ImportError:
    plt = None

# The colours and fonts of the SHOT brand: SHOT red for SHOT's own series, blue for other data, and neutral ink
SHOT_RED, DATA_BLUE, INK, INK_BODY, INK_MUTED = "#E21F25", "#2F6DB5", "#18181B", "#4A4743", "#6B6762"
RULE, FIGURE_FRAME, WHITE = "#BDB8AE", "#DAD6CD", "#FFFFFF"

if plt is not None:
    fonts = {font.name for font in font_manager.fontManager.ttflist}
    SHOT_STYLE = {
        "font.family": [f for f in ("IBM Plex Sans", "Arial", "DejaVu Sans") if f in fonts][:1] or ["sans-serif"],
        "figure.facecolor": WHITE, "axes.facecolor": WHITE,
        "text.color": INK, "axes.labelcolor": INK_BODY, "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
        "axes.edgecolor": RULE, "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False, "legend.fontsize": "small", "legend.labelcolor": INK_MUTED,
        "lines.solid_capstyle": "round", "lines.dash_capstyle": "round",
    }

    with plt.rc_context(SHOT_STYLE):
        # External plots sit on a white card with a thin frame
        figure, (top, bottom) = plt.subplots(2, 1, figsize=(8, 6), sharex=True, edgecolor=FIGURE_FRAME, linewidth=1)

        iterations = [b[0] for b in handler.bounds]
        top.step(iterations, [b[1] if math.isfinite(b[1]) else math.nan for b in handler.bounds], where="post",
                 color=SHOT_RED, linewidth=2, label="primal bound")
        top.step(iterations, [b[2] if math.isfinite(b[2]) else math.nan for b in handler.bounds], where="post",
                 color=DATA_BLUE, linewidth=2, label="dual bound")
        top.axhline(reference["primal"], color=INK_MUTED, linestyle=":", label="optimum of the full problem")
        top.axhline(reduced["primal"], color=INK_MUTED, linestyle="--", label="optimum of the reduced problem")
        top.set_ylim(reduced["primal"] - 2.0, reference["primal"] + 4.0)
        top.set_ylabel("objective")
        top.legend()

        violated = [(i, v) for i, v in handler.dual_points if v > 0.0]
        satisfied = [i for i, v in handler.dual_points if v <= 0.0]
        bottom.semilogy([i for i, _ in violated], [v for _, v in violated], "o", color=DATA_BLUE, label="violated")
        bottom.axhline(FEASIBILITY_TOLERANCE, color=INK_MUTED, linestyle=":", label="feasibility tolerance")

        # A point that satisfies the omitted constraints has no violation to show on the log scale, so it is marked
        # below, as an open dot like a MIP solution in SHOT's figures
        if satisfied:
            bottom.plot(satisfied, [FEASIBILITY_TOLERANCE / 100] * len(satisfied), "o", markerfacecolor=WHITE,
                        markeredgecolor=INK, label="all satisfied")

        bottom.set_xlabel("iteration")
        bottom.set_ylabel("largest violation at the dual points")
        bottom.legend()

        figure.tight_layout()
        plt.show()

## Run 4: adding a black-box primal heuristic

In run 3, SHOT accepts the optimum only once a solution of a dual problem happens to fulfil the omitted constraints.
The points SHOT rejects are often close to feasible, though. A second black box, a heuristic, can turn them into
feasible solutions:

* `black_box_heuristic(point)` is given a point that violates the omitted constraints. It returns a point that it
  expects to fulfil them, or `None` when it cannot repair the point.

The `PrimalCandidateSearch` callback gives the returned points to SHOT with `ctx.addPrimalSolution()`. They are then
primal candidates like any other: `PrimalCandidateCheck` asks the first black box about them, and SHOT checks the
variable bounds, the integrality and the constraints it has. A wrong heuristic can therefore cost time, but it cannot
make SHOT accept an infeasible solution.

**Inside the heuristic** is a repair that only works because of the structure of `ex1223b`: every constraint is
increasing in the continuous variables, which are nonnegative. For each violated omitted constraint, the heuristic
scales its continuous variables down by the smallest factor that fulfils it, found by bisection. Making a continuous
variable smaller never violates another constraint, so the repaired point is feasible if the original point
fulfilled the constraints SHOT has. The heuristic does not change the binary variables, so it declines points where
they are not integral, e.g., the solutions of SHOT's first LP relaxations.

In [ ]:
CONTINUOUS = {INDEX[v] for v, (vtype, _, _) in VARIABLES.items() if vtype == SHOTpy.VariableType.Real}


def black_box_heuristic(point):
    # Scales the continuous variables of each violated omitted constraint down until the constraint is fulfilled
    if any(abs(value - round(value)) > INTEGER_TOLERANCE for i, value in enumerate(point) if i not in CONTINUOUS):
        return None

    repaired = list(point)

    for name in OMITTED:
        if _residual(name, repaired) <= 0.0:
            continue

        linear, squared, _ = CONSTRAINTS[name]
        involved = {INDEX[v] for v in list(linear) + squared} & CONTINUOUS

        def scaled(factor):
            return [value * factor if i in involved else value for i, value in enumerate(repaired)]

        # Not even setting the continuous variables to zero fulfils the constraint
        if _residual(name, scaled(0.0)) > 0.0:
            return None

        low, high = 0.0, 1.0

        for _ in range(60):
            middle = (low + high) / 2

            if _residual(name, scaled(middle)) <= 0.0:
                low = middle
            else:
                high = middle

        repaired = scaled(low)

    return repaired

`LazyConstraintsWithHeuristic` extends the callbacks of run 3. It collects the candidates that `check_candidate`
rejects, except the heuristic's own, and gives them to the heuristic in the next `PrimalCandidateSearch` callback.
In multi-tree mode SHOT calls it once per iteration, after the solutions of the dual problem and the rootsearch have
been checked.

In [ ]:
class LazyConstraintsWithHeuristic(LazyConstraints):
    def __init__(self, oracle, heuristic):
        super().__init__(oracle)
        self.heuristic = heuristic
        self.to_repair = []  # Rejected candidates for the heuristic
        self.submitted = {}  # Points given to SHOT, as tuples, with the iteration

    def register(self, solver):
        super().register(solver)
        solver.registerCallback(SHOTpy.CallbackLocation.PrimalCandidateSearch, self.search)

    def check_candidate(self, ctx):
        number_rejected = len(self.rejected)
        super().check_candidate(ctx)

        if len(self.rejected) > number_rejected and ctx.source != SHOTpy.PrimalSolutionSource.ExternalPrimalSolution:
            self.to_repair.append(list(ctx.point))

    def search(self, ctx):
        points, self.to_repair = self.to_repair, []

        for point in points:
            repaired = self.heuristic(point)

            if repaired is not None and tuple(repaired) not in self.submitted:
                ctx.addPrimalSolution(repaired)
                self.submitted[tuple(repaired)] = ctx.iterationNumber


heuristic_solver, _ = build_solver(OMITTED)
heuristic_handler = LazyConstraintsWithHeuristic(black_box, black_box_heuristic)
heuristic_handler.register(heuristic_solver)

assert heuristic_solver.solveProblem()
with_heuristic = run_summary("with heuristic", heuristic_solver, heuristic_handler)
print_point(with_heuristic)
print(f"  {len(heuristic_handler.submitted)} points from the heuristic, "
      f"{sum(a['source'] == 'ExternalPrimalSolution' for a in heuristic_handler.accepted)} of them accepted")


def first_optimal(handler):
    # The first iteration where SHOT accepted a solution within the gap tolerance of the optimum
    return min(a["iteration"] for a in handler.accepted if a["objective"] <= KNOWN_OPTIMUM + GAP_TOLERANCE)


print(f"\n{'run':<32}{'primal':>11}{'dual':>11}  {'status':<15}{'optimum found':>14}{'iterations':>12}")
for summary, h in ((lazy, handler), (with_heuristic, heuristic_handler)):
    print(f"{summary['run']:<32}{summary['primal']:>11.6f}{summary['dual']:>11.6f}  {summary['status']:<15}"
          f"{first_optimal(h):>14}{h.bounds[-1][0]:>12}")

print("\nAccepted primal solutions with the heuristic:")
for a in heuristic_handler.accepted:
    print(f"  iteration {a['iteration']:>2}: objective {a['objective']:.6f} from {a['source']:<22} "
          f"largest violation {a['violation']:.2e}")

# The heuristic's points were accepted, every accepted solution fulfils the omitted constraints, and the optimum is
# still proven
assert any(a["source"] == "ExternalPrimalSolution" for a in heuristic_handler.accepted)
assert all(a["violation"] <= FEASIBILITY_TOLERANCE for a in heuristic_handler.accepted)
assert with_heuristic["status"] == "OptimalGlobal" and with_heuristic["gap"] <= GAP_TOLERANCE
assert abs(with_heuristic["primal"] - KNOWN_OPTIMUM) <= GAP_TOLERANCE
assert with_heuristic["dual"] <= KNOWN_OPTIMUM + 1e-6

With the heuristic, SHOT has the optimum several iterations earlier: the heuristic repairs a rejected solution of a
dual problem that has the optimal binary variables, by lowering $x_3$ until $e_9$ holds. The gap closes only when the
dual bound has caught up, so the total number of iterations need not change.

The figure plots the objective over the iterations: the primal bound without the heuristic (dashed) and with it
(solid), and the dual bound of the run with the heuristic. The solid primal bound reaches the optimum earlier.

In [ ]:
if plt is not None:
    with plt.rc_context(SHOT_STYLE):
        figure, axes = plt.subplots(figsize=(8, 3.5), edgecolor=FIGURE_FRAME, linewidth=1)

        def bound(bounds, which):
            return [b[which] if math.isfinite(b[which]) else math.nan for b in bounds]

        axes.step([b[0] for b in handler.bounds], bound(handler.bounds, 1), where="post", color=INK_MUTED,
                  linestyle="--", linewidth=2, label="primal bound, run 3")
        axes.step([b[0] for b in heuristic_handler.bounds], bound(heuristic_handler.bounds, 1), where="post",
                  color=SHOT_RED, linewidth=2, label="primal bound with the heuristic")
        axes.step([b[0] for b in heuristic_handler.bounds], bound(heuristic_handler.bounds, 2), where="post",
                  color=DATA_BLUE, linewidth=2, label="dual bound with the heuristic")
        axes.axhline(reference["primal"], color=INK_MUTED, linestyle=":", label="optimum of the full problem")
        axes.set_ylim(reduced["primal"] - 0.5, reference["primal"] + 2.5)
        axes.set_xlabel("iteration")
        axes.set_ylabel("objective")
        axes.legend()

        figure.tight_layout()
        plt.show()

## Limitations and variations

* **Valid cuts.** The callbacks trust the black box that its cuts remove no feasible solution; they can only check
  that a cut removes the point it was generated in. The ECP cuts here are valid because the omitted constraints are
  convex. For a nonconvex constraint a linearization can cut off feasible points, and the dual bound would no longer
  be valid.
* **Primal solutions.** SHOT's primal heuristics, such as the fixed-integer NLP problems and the rootsearch, only know
  the reduced problem. Their solutions are rejected when they violate omitted constraints. Without a heuristic of its
  own, SHOT finds its solutions among the points of the dual problems once enough cuts have been added, which can
  take long for other problems. The repair heuristic of run 4 depends on the structure of `ex1223b`; another problem
  needs a heuristic of its own, e.g., an NLP problem solved with the omitted constraints.
* **Strategies and solvers.** The example uses the multi-tree strategy with HiGHS or Cbc. The single-tree strategy,
  and the direct MIQCQP strategy SHOT may use with Gurobi or CPLEX, are not covered.
* **Another black box.** Any function with the same contract can replace `black_box`, and `LazyConstraints` works
  unchanged. Rerun the notebook from the start after changing it.
* **Changing the omitted constraints.** `OMITTED` can be any subset of `CONSTRAINTS`; for example `("e2", "e7", "e8",
  "e9")` leaves out all the quadratic constraints. A linear constraint can also be omitted, in which case its cut is
  the constraint itself. If the solution of the reduced problem already satisfies the omitted constraints, nothing is
  rejected and the checks of the reduced problem and the cuts fail on purpose, since the example then shows nothing.